In [73]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Input

In [74]:
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

data = df[['Price']].values

In [75]:
split_raw = int(len(data) * 0.8)

train_data = data[:split_raw]
test_data = data[split_raw - 60:] 

In [76]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [77]:
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

time_step = 60

X_train_lstm, y_train_lstm = create_sequences(train_scaled, time_step)
X_test_lstm, y_test_lstm = create_sequences(test_scaled, time_step)

X_train_lstm = X_train_lstm.reshape(X_train_lstm.shape[0], time_step, 1)
X_test_lstm = X_test_lstm.reshape(X_test_lstm.shape[0], time_step, 1)

In [79]:
lstm_pred = model.predict(X_test_lstm)
lstm_pred = scaler.inverse_transform(lstm_pred).flatten()

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


In [86]:
model = Sequential()
model.add(Input(shape=(time_step, 1)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.fit(
    X_train_lstm, y_train_lstm,
    epochs=20,
    batch_size=32,
    verbose=1
)

Epoch 1/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0091
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.5427e-04
Epoch 3/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.6145e-04
Epoch 4/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.2296e-04
Epoch 5/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.2007e-04
Epoch 6/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.1059e-04
Epoch 7/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.0587e-04
Epoch 8/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 3.7567e-04
Epoch 9/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.6691e-04
Epoch 10/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.6395e-04
Epoch 11/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4.0313e-04
Epoch 12/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 3.2583e-04
Epoch 13/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.1768e-04
Epoch 14/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.3259e-04
Epoch 15/20
76/76 ━

In [80]:
hybrid_df = df.iloc[split_raw - time_step:].reset_index(drop=True)
hybrid_df = hybrid_df.iloc[time_step:].reset_index(drop=True)

In [81]:
hybrid_df = df.iloc[split_raw - time_step:].reset_index(drop=True)

hybrid_df = hybrid_df.iloc[time_step:].reset_index(drop=True)

lstm_pred = lstm_pred[-len(hybrid_df):]

hybrid_df['LSTM_Pred'] = lstm_pred

In [82]:
hybrid_df['Lag1'] = hybrid_df['Price'].shift(1)

hybrid_df = hybrid_df.dropna().reset_index(drop=True)


In [83]:
X = hybrid_df[['Lag1', 'LSTM_Pred']]
y = hybrid_df['Price'] - hybrid_df['Lag1']

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

actual = hybrid_df['Price'].iloc[split:].values

In [84]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

pred_diff = rf.predict(X_test)

final_pred = X_test['Lag1'].values + pred_diff

In [85]:
rmse = np.sqrt(mean_squared_error(actual, final_pred))
mae = mean_absolute_error(actual, final_pred)
r2 = r2_score(actual, final_pred)
mape = np.mean(np.abs((actual - final_pred) / actual)) * 100

print("\n=== HYBRID MODEL RESULTS ===")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")
print(f"MAPE: {mape:.6f}")


=== HYBRID MODEL RESULTS ===
MAE: 2005.224099
RMSE: 2290.475729
R2: 0.967549
MAPE: 1.706083


In [71]:
import os

os.makedirs("models", exist_ok=True)

In [72]:
import joblib
joblib.dump(rf, "models/hybrid_rf.pkl")

['models/hybrid_rf.pkl']